In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Vertex AI Model Garden - Batch Prediction Example

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/notebooks/deploy-notebook?download_url=https://github.com/google-research/remote-sensing/raw/refs/heads/main/remote_sensing/vertex_ai/notebooks/vlm_batch_prediction_example.ipynb">
      <img alt="Workbench logo" src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" width="32px"><br> Run in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw%2Egithubusercontent%2Ecom%2Fgoogle%2Dresearch%2Fremote%2Dsensing%2Fmaster%2Fremote%5Fsensing%2Fvertex%5Fai%2Fnotebooks%2Fvlm%5Fbatch%5Fprediction%5Fexample%2Eipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/notebooks/model_garden_remote_sensing_deployment.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
  </td>
</tr></tbody></table>

# RS Imagery Batch Inference on VertexAI

This notebook shows how to run a Batch Prediction Job deployed VLMs (image and text) on Vertex AI.

**Prepare the environment for interacting with Vertex AI:**

Initialize the Vertex AI SDK using the aiplatform.init() function.

Configure the SDK to work with your specific Google Cloud project (PROJECT_ID) and region (REGION) that were defined in the previous configuration cell. This step is necessary before using other SDK functions to manage Vertex AI resources.

In [ ]:
# @title Setup Notebook

import base64
import io
import json
from typing import Any

from google.cloud import aiplatform
from google.cloud import storage
from PIL import Image


def to_png_bytes(img: Image.Image) -> bytes:
  """Encodes the `img` as PNG bytes."""
  buf = io.BytesIO()
  img.save(buf, format="PNG")
  return buf.getvalue()


def to_b64_png(img: Image.Image) -> str:
  """Converts the `img` to a b64 encoded PNG bytes."""
  return base64.b64encode(to_png_bytes(img)).decode()


def write_jsonl_instances(
    bucket: storage.Bucket, path: str, instances: list[dict[str, Any]]
):
  """Writes the list of instances (dicts) as a JSONL serialized file.

  Each dict is an inference instance, matching one of the following structures:

    {'image': <b64 image>} - Image inference only.
    {'text': <str>} - Text inference only.
    {'image': <b64 image, 'texts': list<str>} - Image & text inference.
  """
  with bucket.blob(path).open("wt") as f:
    f.writelines(f"{json.dumps(instance)}\n" for instance in instances)


# @markdown Replace the constants below with your project/region config
PROJECT_ID = ""  # @param { type : 'string' }
REGION = "us-central1"  # @param { type : 'string' }
# Initialize Vertex AI SDK
aiplatform.init(project=PROJECT_ID, location=REGION)

In [ ]:
# @title Initialize data bucket
# @markdown ### Enter a GCS bucket name and a path within the bucket:

BUCKET_NAME = ""  # @param { type : 'string' }
OUTPUT_PATH = "batch_inference/inputs"  # @param { type : 'string' }

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

# Download sample image
!wget -O harbor.jpg https://mrsg.aegean.gr/images/uploads/it2zi0eidej4ql33llj.jpg
harbor_img = Image.open("harbor.jpg")

In [ ]:
# @title Prepare data: Image list file

input_uris = []

# The sample image (harbor) is replicated 10 times as an example.
for i in range(10):
  img_path = f"{OUTPUT_PATH}/images/img{i}.png"
  input_uris.append(f"gs://{BUCKET_NAME}/{img_path}")
  bucket.blob(img_path).upload_from_string(
      to_png_bytes(harbor_img), content_type="image/png"
  )

with bucket.blob(f"{OUTPUT_PATH}/input_uris.txt").open("wt") as f:
  f.writelines([f"{i}\n" for i in input_uris])

In [ ]:
# @title Prepare JSONL input

# This cell generates sample inputs (JSONL files) in 3 formats:
# Image, text and image_text, the input files are sharded to optionally reduce
# the size of each file. The file pattern (with wildcards) is used as input for
# the batch pipeline, e.g. "gs://bucket_path/image*.jsonl"

# Write 3 shards of text input instances (10 instances each).
instances = [{"text": "test string"}] * 10
for i in range(3):
  write_jsonl_instances(bucket, f"{OUTPUT_PATH}/text{i}.jsonl", instances)

# Write 10 image input instances into 3 shards.
instances = [{"image": to_b64_png(harbor_img)}] * 10
for p in range(3):
  write_jsonl_instances(bucket, f"{OUTPUT_PATH}/image{p}.jsonl", instances)

# Write 10 image & texts input instances into a single file.
instances = [
    {"image": to_b64_png(harbor_img), "texts": ["text1", "text2"]},
] * 10
write_jsonl_instances(bucket, f"{OUTPUT_PATH}/combined.jsonl", instances)

In [ ]:
# @title Run batch inference on JSONL data input

JOB_DISPLAY_NAME = "batch-inference-vlm-test"  # @param { type: 'string' }
# @markdown Enter the project number and model id, the project number is not the
# @markdown same as project id (it can be found in the project settings).
PROJECT_NUMBER = "<project_number_here>"  # @param { type: 'string' }
MODEL_ID = "<model_id_here>"  # @param { type: 'string' }
MODEL_RESOURCE_NAME = (
    f"projects/{PROJECT_NUMBER}/locations/{REGION}/models/{MODEL_ID}"
)

# @markdown Choose the input (instances) format, either a file-list of images or
# @markdown a JSONL file pattern of JSON formatted inputs.
INPUT_SOURCE_FORMAT = "file-list"  # @param["file-list", "jsonl"]
# @markdown Configure batch input source  This can use string wildcards such as
# @markdown '*' and '?' to support sharded inputs.
INPUT_SOURCE_PATTERN =  "gs://<bucket>/batch_inference/inputs/inputlist.txt" # @param { type: 'string' }
# @markdown Configure the output folder path, predictions will be written here.
GCS_OUTPUT_PATH = "gs://<bucket>/batch_inference/outputs"  # @param { type: 'string' }
# @markdown Configure the batch runtime setup
USE_GPU = True  # @param { type: 'boolean' }
BATCH_SIZE = 16  # @param { type: 'number' }
REPLICA_COUNT = 1  # @param { type: 'number' }
MAX_REPLICA_COUNT = 4  # @param { type: 'number' }

machine_type = "g2-standard-8"
if USE_GPU:
  accelerator_type = "NVIDIA_L4"
  accelerator_count = 1
else:
  accelerator_type = None
  accelerator_count = None

model = aiplatform.Model(MODEL_RESOURCE_NAME)

job = model.batch_predict(
    job_display_name=JOB_DISPLAY_NAME,
    gcs_source=INPUT_SOURCE_PATTERN,
    gcs_destination_prefix=GCS_OUTPUT_PATH,
    instances_format=INPUT_SOURCE_FORMAT,
    machine_type=machine_type,
    accelerator_count=accelerator_count,
    accelerator_type=accelerator_type,
    starting_replica_count=REPLICA_COUNT,
    max_replica_count=MAX_REPLICA_COUNT,
    labels={
        "task": "batch-inference",
        "vertex-ai-pipelines-run-billing-id": JOB_DISPLAY_NAME,
    },
    batch_size=BATCH_SIZE,
    sync=False,
)

print(f"Batch prediction job started: {job}")

In [ ]:
# Monitor the job status

print(
    f"Running batch prediction {job.display_name}, resource:"
    f" {job.resource_name}. State: {job.state}"
)